# Traffic Signal Control DRL: Main Training Pipeline (Roundabout Network)

This notebook provides the complete, end-to-end training pipeline for Deep Reinforcement Learning (DQN and PPO) agents controlling traffic lights on the official roundabout network.

### Key Components:
1. **Google Colab Environment Setup**: Clones repository, installs Eclipse SUMO binaries and dependencies.
2. **Scenario Generation Pipeline**: Builds realistic mixed-traffic scenarios with motorbikes, cars, MPVs, buses, trucks, and emergency vehicles from `selected_scenario.csv`.
3. **Network Topology Inspection**: Connects directly to `sumo/net/official/roundabout.net.xml`:
   - **5 Inbound Approach Edges**: `A1_A`, `A2_A`, `A3_A`, `A4_A`, `A6_A` (10 total approach lanes: `A1_A_0`, `A1_A_1`, etc.)
   - **6 Internal Circulatory Segments**: `E13`, `E16`, `E46`, `E24`, `E25`, `E35`
   - **Central Traffic Light**: `traffic_light` (3 green phases `rrGGGGrrGG`, `rrrrGGGGrr`, `GGGGrrrrrr`)
4. **Custom Mixed Traffic Observation**: `MixedTrafficObservation` tracking approach PCU density, per-class queues, ring segment speeds, active phase, and elapsed green ratio.
5. **Multi-Objective Reward**: `CombinedReward` balancing delay ($W_t$), physical queues ($Q_t$), flicker suppression ($P_\text{flicker}$), and deadlock penalties ($P_\text{deadlock}$).
6. **Training DQN & PPO**: Config-driven model building, leak-free `ScenarioSampler` on `TR` split, `RobustCheckpointCallback`, and evaluation visualization.

## 1. Google Colab Environment Setup

In Google Colab, we clone the repository, change into the project root directory, install Eclipse SUMO and its Python bindings, and install all required project dependencies.

In [ ]:
# Cell 1: Resolve the project root and prepare the local/Colab environment
from pathlib import Path
import os
import sys

project_root = Path.cwd()

if "google.colab" in sys.modules:
    repo_dir = Path("/content/DRL-Traffic-Signal-Control")
    if not repo_dir.exists():
        !git clone https://github.com/TCL03-HCMUT/DRL-Traffic-Signal-Control.git /content/DRL-Traffic-Signal-Control
    project_root = repo_dir
    %cd "$project_root"
else:
    candidates = [
        Path.cwd(),
        Path.cwd().parent,
        Path.home() / "DRL-Traffic-Signal-Control",
    ]
    for candidate in candidates:
        if (candidate / "src" / "traffic_drl").exists():
            project_root = candidate
            break
    os.chdir(project_root)

print(f"Project root: {project_root}")
print(f"Repository structure present: {(project_root / 'src' / 'traffic_drl').exists()}")


In [ ]:
# Cell 2: Install SUMO and Python dependencies only when needed
import os
import sys

if "google.colab" in sys.modules:
    !sudo add-apt-repository -y ppa:sumo/stable
    !sudo apt-get update -y
    !sudo apt-get install -y sumo sumo-tools sumo-doc
    !pip install -r requirements-colab.txt
    !pip install -e .
else:
    print("Running in a local project environment. Using the repository's existing Python environment.")
    print("If SUMO is missing, install it via the project setup instructions or the requirements file.")


In [ ]:
# Cell 3: Configure SUMO_HOME and verify environment
import os
import sys
from pathlib import Path
import torch
import traci
import sumo_rl
import gymnasium as gym
import stable_baselines3

if "google.colab" in sys.modules:
    os.environ["SUMO_HOME"] = "/usr/share/sumo"
elif "SUMO_HOME" not in os.environ:
    candidates = [
        "/usr/share/sumo",
        "C:/Program Files (x86)/Eclipse/Sumo",
        Path.cwd() / "sumo",
        Path.cwd().parent / "sumo",
    ]
    for candidate in candidates:
        path = Path(candidate)
        if path.exists():
            os.environ["SUMO_HOME"] = str(path)
            break

print("Environment Verification:")
print(f"  Project root: {Path.cwd()}")
if not str(Path.cwd()).isascii():
    print("  WARNING: project path contains non-ASCII characters. On Windows, SUMO may fail to access route files from this location. Move the repo to an ASCII-only path like C:/traffic_drl.")
print(f"  SUMO_HOME: {os.environ.get('SUMO_HOME')}")
print(f"  PyTorch version: {torch.__version__} (CUDA available: {torch.cuda.is_available()})")
print(f"  Gymnasium version: {gym.__version__}")
print(f"  Stable-Baselines3 version: {stable_baselines3.__version__}")
print(f"  SUMO TraCI version: {traci.__version__}")
print(f"  SUMO-RL imported from: {sumo_rl.__file__}")

assert Path.cwd().joinpath("src", "traffic_drl").exists(), "Project root is not set correctly."


In [ ]:
# Cell 4: Verify sumo CLI command
!sumo --version


## 2. Scenario Generation Setup

The project generates realistic mixed-traffic scenarios driven by manifest specifications:
- `scenarios/json_definitions/definition.jsonc`: Vehicle profiles (electric/gas motorbikes, cars, MPVs, buses, trucks, ambulances).
- `scenarios/json_definitions/routes.jsonc`: Valid roundabout turning movements between approach legs.
- `scenarios/json_definitions/scenario_manifest.jsonc`: Approach hourly demands, vehicle class mix, and turning ratios.
- `scenarios/json_definitions/selected_scenario.csv`: Formal 30-scenario train (`TR`), validation (`VA`), and test (`TE`) catalog.

Here we invoke the scenario generator to populate `scenarios/sumo_definitions/` with the required `.rou.xml` route files and `vtypes.add.xml`.

In [ ]:
# Cell 4: Generate all scenarios from selected_scenario.csv
import csv
import subprocess
from pathlib import Path

csv_path = Path("scenarios/json_definitions/selected_scenario.csv")
factory_script = Path("scenarios/generators/scenario_factory.py")

print(f"Reading scenarios from {csv_path}...")
with open(csv_path, "r", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    scenarios_to_generate = [row["scenario_id"] for row in reader if row.get("scenario_id")]

print(f"Found {len(scenarios_to_generate)} scenarios across TR, VA, and TE splits.")
print("Generating scenarios into scenarios/sumo_definitions/...")

for sid in scenarios_to_generate:
    result = subprocess.run(
        [sys.executable, str(factory_script), "--scenario", sid],
        capture_output=True,
        text=True,
    )
    if result.returncode != 0:
        print(f"  Error generating {sid}: {result.stderr}")
    else:
        print(f"  [OK] {sid}")

print("Scenario generation complete!")


In [ ]:
# Cell 5: Inspect generated files in scenarios/sumo_definitions/
sumo_def_dir = Path("scenarios/sumo_definitions")
generated_routes = list(sumo_def_dir.glob("*.rou.xml"))
vtypes_file = sumo_def_dir / "vtypes.add.xml"

print(f"Total route files generated: {len(generated_routes)}")
print(f"Vehicle types definition file exists: {vtypes_file.exists()} ({vtypes_file})")

# Display first 5 generated route files
for r in generated_routes[:5]:
    print(f"  - {r.name} ({r.stat().st_size} bytes)")


## 3. Roundabout Network Topology & Edge/Lane IDs

Examining the network geometry defined in `sumo/net/official/roundabout.net.xml`:

| Component | Identifiers | Description |
| :--- | :--- | :--- |
| **Traffic Light ID** | `traffic_light` | Single centralized junction controlling all inbound entry points |
| **Inbound Approaches ($N=5$)** | `A1_A`, `A2_A`, `A3_A`, `A4_A`, `A6_A` | Entry edges from surrounding arterials (West, East, NW, SE, SW) |
| **Approach Lanes (10 total)** | `A1_A_0`, `A1_A_1`, `A2_A_0`, `A2_A_1`, `A3_A_0`, `A3_A_1`, `A4_A_0`, `A4_A_1`, `A6_A_0`, `A6_A_1` | 2 lanes per approach |
| **Circulatory Ring ($M=6$)** | `E13`, `E16`, `E46`, `E24`, `E25`, `E35` | Clockwise circular ring segments around junctions J1–J6 |
| **Outbound Legs** | `A1_B`, `A2_B`, `A3_B`, `A4_B`, `A5`, `A6_B` | Exit edges carrying traffic away from the roundabout |
| **Green Phase Stages ($K=3$)** | 0: `rrGGGGrrGG`, 2: `rrrrGGGGrr`, 4: `GGGGrrrrrr` | Non-conflicting green phases separated by yellow transitions |

In [ ]:
# Cell 6: Inspect network file XML attributes directly
import xml.etree.ElementTree as ET

net_file = Path("sumo/net/official/roundabout.net.xml")
assert net_file.exists(), f"Network file not found: {net_file}"

tree = ET.parse(net_file)
root = tree.getroot()

# Extract roundabout ring edges
roundabout_elem = root.find("roundabout")
if roundabout_elem is not None:
    print("Roundabout element in network:")
    print("  Nodes:", roundabout_elem.get("nodes"))
    print("  Ring Edges:", roundabout_elem.get("edges"))

# Extract traffic light logic
tl_elem = root.find("tlLogic")
if tl_elem is not None:
    print(f"\nTraffic Light ID: '{tl_elem.get('id')}' (programID: '{tl_elem.get('programID')}')")
    phases = tl_elem.findall("phase")
    print(f"  Total defined phases: {len(phases)}")
    for idx, p in enumerate(phases):
        print(f"    Phase {idx}: duration={p.get('duration')}s, state='{p.get('state')}'")

# Extract inbound approaches with connections to traffic_light
tl_connections = root.findall("connection[@tl='traffic_light']")
inbound_edges = sorted(list(set(c.get("from") for c in tl_connections if c.get("from"))))
print(f"\nInbound approaches with traffic signal links ({len(inbound_edges)}):", inbound_edges)


## 4. Load Configurations (Environment, DQN, PPO)

We now load the centralized typed configuration files:
- `configs/environment/roundabout.yaml`: Defines simulation timing (3600s, $\Delta t=5$s, $t_\text{min}=10$s, $t_\text{yellow}=3$s), `roundabout.net.xml`, `vtypes.add.xml`, approach IDs, and ring segment IDs.
- `configs/train/dqn_roundabout.yaml`: Training hyperparameters for DQN, reward weights for `CombinedReward`, and replay buffer sizing.
- `configs/train/ppo_roundabout.yaml`: Training hyperparameters for PPO, rollout step horizon, and entropy coefficient.

In [ ]:
# Cell 7: Select the algorithm and load the matching training config
from traffic_drl.config import load_env_config, load_train_config

SELECTED_ALGO = "DQN"  # Change to "PPO" to run the PPO setup instead
cfg_by_algo = {
    "DQN": load_train_config("configs/train/dqn_roundabout.yaml"),
    "PPO": load_train_config("configs/train/ppo_roundabout.yaml"),
}
active_config = cfg_by_algo[SELECTED_ALGO]
env_config = load_env_config(active_config.environment.env_config_path)

print(f"Selected algorithm: {SELECTED_ALGO}")
print("Environment Config Loaded:")
print("  Network:", env_config.network.net_file)
print("  Traffic Light ID:", env_config.traffic_light.ts_id)
print("  Timing: delta_time=", env_config.timing.delta_time, "min_green=", env_config.timing.min_green, "max_green=", env_config.timing.max_green)
print("  Approach IDs:", env_config.observation_features.approach_ids)
print("  Ring Segment IDs:", env_config.observation_features.ring_segment_ids)
print("  Vehicle Classes:", env_config.observation_features.vehicle_classes)

print("\nActive Training Config Loaded:")
print("  Experiment:", active_config.experiment.name, "| Seed:", active_config.experiment.seed)
print("  Reward Type:", active_config.reward.type, "| Params:", active_config.reward.params)
print("  Timesteps:", active_config.training_control.total_timesteps, "| Save Freq:", active_config.training_control.save_freq)


## 5. Create Environment with Custom Observation & Reward

We build the Gymnasium environment via `make_vectorized_environment`:
- **Leak-free sampling**: Wrapped with `ScenarioSampler` restricted exclusively to the `TR` split.
- **Observation**: `MixedTrafficObservation` mapping $N=5$ approaches, $C=4$ classes, $M=6$ ring segments, and $K=3$ green phases into a normalized 35-dimensional vector $[0, 1]^{35}$.
- **Reward**: `CombinedReward` computing delay, queue, flicker, and deadlock terms.
- **Normalization**: `VecNormalize` wrapping observation and reward channels.

In [ ]:
# Cell 8: Build the training environment using the active algorithm config
from traffic_drl.environment.scenario_factory import ScenarioManifest
from traffic_drl.train.scenario_sampler import ScenarioSampler
from traffic_drl.environment.make_env import make_vectorized_environment, build_reward_fn

# Load the scenario manifest catalog from the active config
manifest = ScenarioManifest.from_csv(active_config.environment.manifest_path)
print(f"Loaded manifest with {len(manifest.records)} total records.")

# Create scenario sampler restricted strictly to the active split
sampler = ScenarioSampler(
    manifest=manifest,
    allowed_split=active_config.experiment.split_allowed,
    seed=active_config.experiment.seed,
)
initial_record = sampler.sample()
print(f"Initial training route sampled: {initial_record.scenario_id} -> {initial_record.route_file}")

# Resolve the reward function from the active config
reward_fn = build_reward_fn(active_config.reward)
print(f"Reward callable constructed: {type(reward_fn).__name__}")

# Create vectorised training environment using active_config values
train_env = make_vectorized_environment(
    config=env_config,
    route_file=initial_record.route_file,
    run_id=active_config.experiment.name,
    base_dir=Path("outputs/runs"),
    seed=active_config.experiment.seed,
    reward_fn=reward_fn,
    wrap=True,
    scenario_sampler=sampler,
    custom_observation=True,
    training=True,
    norm_obs=active_config.normalisation.norm_obs,
    norm_reward=active_config.normalisation.norm_reward,
    clip_obs=active_config.normalisation.clip_obs,
    clip_reward=active_config.normalisation.clip_reward,
    approach_ids=env_config.observation_features.approach_ids,
    ring_segment_ids=env_config.observation_features.ring_segment_ids,
    vehicle_classes=env_config.observation_features.vehicle_classes,
)

print("\nVectorized Environment Ready:")
print(f"  Observation Space: {train_env.observation_space}")
print(f"  Action Space: {train_env.action_space}")


In [ ]:
# Cell 9: Test reset and step to verify MDP compliance
obs = train_env.reset()
print("Reset succeeded. Initial observation shape:", obs.shape)
print("Observation values preview:", obs[0][:10])

# Take one test step with action 0
test_action = [0]
next_obs, reward, done, info = train_env.step(test_action)
print("\nTest Step Results:")
print(f"  Reward: {reward[0]:.4f}")
print(f"  Done: {done[0]}")
print(f"  Active scenario: {info[0].get('scenario')}")


## 6. Build Models & Checkpoint Callbacks

We configure our training callbacks:
- `RobustCheckpointCallback`: Persists model weights, replay buffer, `VecNormalize` running statistics, and RNG state into `best/` and `last/` directories.
- `TrafficMetricsCallback`: Records throughput, waiting time, queue lengths, and min-green violations into SB3 logs.

In [ ]:
# Cell 10: Setup callbacks and build the model
from stable_baselines3.common.callbacks import CallbackList
from traffic_drl.train.callbacks import RobustCheckpointCallback, TrafficMetricsCallback
from traffic_drl.train.train_dqn import build_dqn_model, train_dqn
from traffic_drl.train.train_ppo import build_ppo_model, train_ppo

# Use the algorithm selected above for all downstream setup.
active_config = cfg_by_algo[SELECTED_ALGO]
checkpoint_dir = Path("outputs/runs") / active_config.experiment.name / "checkpoints"
tb_dir = Path("outputs/runs") / active_config.experiment.name / "tensorboard"

checkpoint_callback = RobustCheckpointCallback(
    save_freq=active_config.training_control.save_freq,
    save_dir=checkpoint_dir,
    save_replay_buffer=(SELECTED_ALGO == "DQN"),
    save_vec_normalize=True,
    verbose=1,
)
metrics_callback = TrafficMetricsCallback()
callbacks = CallbackList([checkpoint_callback, metrics_callback])

# Build model from the active config
if SELECTED_ALGO == "DQN":
    print("Building DQN Model...")
    model = build_dqn_model(
        env=train_env,
        config=active_config,
        seed=active_config.experiment.seed,
        tensorboard_log=tb_dir,
    )
else:
    print("Building PPO Model...")
    model = build_ppo_model(
        env=train_env,
        config=active_config,
        seed=active_config.experiment.seed,
        tensorboard_log=tb_dir,
    )

print(f"{SELECTED_ALGO} model successfully built:")
print(f"  Policy: {model.policy}")
print(f"  Device: {model.device}")


## 7. Main Training Execution

We train the model for the configured timesteps (default 100,000 steps). You can adjust `total_timesteps` for smoke-test or full training runs.

In [ ]:
# Cell 11: Execute training loop using the active config timesteps
# Use the value defined in the selected algorithm's config instead of a hard-coded override.
TRAIN_TIMESTEPS = active_config.training_control.total_timesteps

print(f"Starting {SELECTED_ALGO} training for {TRAIN_TIMESTEPS} timesteps...")
if SELECTED_ALGO == "DQN":
    trained_model = train_dqn(model, total_timesteps=TRAIN_TIMESTEPS, callback=callbacks)
else:
    trained_model = train_ppo(model, total_timesteps=TRAIN_TIMESTEPS, callback=callbacks)

print("Training complete!")

# Close environment cleanly to terminate SUMO subprocesses
train_env.close()
print("Environment closed successfully.")


## 8. Checkpoints and Training Monitoring

Inspect the saved checkpoints and normalisation statistics in the experiment run folder.

In [ ]:
# Cell 12: Verify checkpoint files
print(f"Inspecting checkpoints at: {checkpoint_dir}")
if checkpoint_dir.exists():
    for path in sorted(checkpoint_dir.rglob("*")):
        if path.is_file():
            rel = path.relative_to(checkpoint_dir)
            print(f"  - {rel} ({path.stat().st_size} bytes)")
else:
    print("Checkpoint directory not yet created.")


In [ ]:
# Cell 13: Optional TensorBoard dashboard launcher (Colab & Local)
%load_ext tensorboard
%tensorboard --logdir outputs/runs
